In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import os

df = pd.read_csv('../data/processed/cleaned.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
def generate_doc(row):
    # Convert log price back to dollars
    price = int(np.expm1(row['SalePrice']))
    
    return f"""
    A {int(row['Gr Liv Area'])} sq ft home in the {row['Neighborhood']} 
    neighborhood sold for ${price:,}. The property has an Overall Quality 
    rating of {int(row['Overall Qual'])} out of 10, built in {int(row['Year Built'])}. 
    It features {int(row['Full Bath'])} full bathrooms, {int(row['Bedroom AbvGr'])} 
    bedrooms, and a {int(row['Garage Cars'])}-car garage. 
    The kitchen quality is rated {row['Kitchen Qual']} and the 
    basement size is {int(row['Total Bsmt SF'])} sq ft.
    """.strip()

documents = df.apply(generate_doc, axis=1).tolist()

print(f"Generated {len(documents)} documents")
print("\nSample document:")
print(documents[0])

In [ ]:
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding documents... (this may take 2-3 minutes)")
embeddings = embedder.encode(documents, show_progress_bar=True)

print(f"\nEmbedding shape: {embeddings.shape}")
# Should be (2927, 384) — 2927 documents, 384 dimensions per embedding

In [ ]:
dimension = embeddings.shape[1]  # 384

index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))

print(f"FAISS index built")
print(f"Total vectors in index: {index.ntotal}")

In [ ]:
# Test query — describe a house and find similar ones
test_query = "3 bedroom house in CollgCr neighborhood with good kitchen quality and 2 car garage"

query_embedding = embedder.encode([test_query])
distances, indices = index.search(query_embedding.astype('float32'), k=5)

print("Top 5 most similar houses to query:")
print(f"Query: {test_query}\n")
for i, idx in enumerate(indices[0]):
    print(f"--- Match {i+1} (distance: {distances[0][i]:.4f}) ---")
    print(documents[idx])
    print()

In [ ]:
os.makedirs('../data/vector_store', exist_ok=True)

# Save FAISS index
faiss.write_index(index, '../data/vector_store/houses.index')

# Save documents and embeddings for reference
with open('../data/vector_store/documents.pkl', 'wb') as f:
    pickle.dump(documents, f)

with open('../data/vector_store/embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)

print("Vector store saved")